# Manufacturing AI Agent — LangGraph 멀티 에이전트 구현 (LLM 통합)  — **v0.2**

## v0.2 변경 이력
- 🐛 **버그 수정**: `numpy.float64` is not msgpack serializable 에러 해결
  - 원인: `PredictionService.ml_predict()`가 반환한 `top_feature_importances` dict 안에 numpy.float64 값이 그대로 들어감. `PredictionResult.ml_prediction: Optional[dict]`은 Pydantic이 내부 타입을 변환하지 않아 그대로 state로 흘러갔고, LangGraph checkpointer가 msgpack 직렬화 시 실패.
  - 수정 1: `ml_predict()`에서 모든 numpy 값을 `float()`로 명시 변환
  - 수정 2: 방어용 `to_jsonable()` 헬퍼 추가 → 모든 agent 노드 반환 직전에 적용

본 노트북은 `README.md`의 설계를 LangGraph로 구현한 참조 구현입니다.

**핵심 구성**
- **프레임워크**: LangGraph (StateGraph + Conditional Edges)
- **단기 메모리**: `MemorySaver` Checkpointer (thread 단위 state 보존)
- **장기 메모리**: `SqliteSaver` / 자체 `SqliteLongTermMemory` (cross-session 영속화)
- **벡터 스토어**: ChromaDB (PersistentClient)
- **PredictionAgent**: RandomForest + class_weight (sklearn) — 결정론적 ML
- **EvidenceAgent / SafetyAgent / FinalAnswerNode**: LLM (OpenAI) 기반
- **Supervisor**: 결정론적 라우터

## 0. 환경 설정 & 의존성 설치

In [ ]:
# 최초 1회만 실행
# %pip install -q langgraph langchain langchain-openai langchain-community chromadb pydantic pandas scikit-learn numpy langgraph-checkpoint-sqlite python-dotenv

In [ ]:
from __future__ import annotations

import json
import os
import re
import sqlite3
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Optional, TypedDict

import numpy as np
import pandas as pd
from pydantic import BaseModel, Field

ROOT = Path('.').resolve()
DATA_DIR = ROOT / 'data'
ARTIFACT_DIR = ROOT / 'artifacts'
ARTIFACT_DIR.mkdir(exist_ok=True)

CHECKPOINT_DB = str(ARTIFACT_DIR / 'checkpoints.sqlite')
LONGTERM_DB = str(ARTIFACT_DIR / 'longterm.sqlite')
CHROMA_PATH = str(ARTIFACT_DIR / 'chroma')

print('ROOT:', ROOT)

In [ ]:
# v0.2 추가 — LangGraph checkpointer가 msgpack으로 state를 직렬화할 때
# numpy 스칼라/array가 들어있으면 'is not msgpack serializable' 에러가 난다.
# agent 노드가 state로 돌려보내는 dict는 모두 이 함수를 거쳐 Python 원시 타입으로 변환한다.

def to_jsonable(obj):
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, dict):
        return {k: to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    return obj

In [ ]:
# v0.2 — 비밀값과 모델 설정은 .env 파일에서 로드한다.
# .env 가 없으면 .env.example 을 fallback 으로 시도한다.
from dotenv import load_dotenv

ENV_FILE = ROOT / '.env'
ENV_EXAMPLE = ROOT / '.env.example'

if ENV_FILE.exists():
    load_dotenv(ENV_FILE, override=False)
    print(f'[env] loaded: {ENV_FILE.name}')
elif ENV_EXAMPLE.exists():
    load_dotenv(ENV_EXAMPLE, override=False)
    print(f'[env] loaded fallback: {ENV_EXAMPLE.name}')
else:
    print('[env] no .env file found — using process env only')

LLM_MODEL = os.environ.get('LLM_MODEL', 'gpt-4o-mini')
LLM_TEMPERATURE = float(os.environ.get('LLM_TEMPERATURE', '0'))
OPENAI_KEY_SET = bool(os.environ.get('OPENAI_API_KEY'))

from langchain_openai import ChatOpenAI

LLM = ChatOpenAI(model=LLM_MODEL, temperature=LLM_TEMPERATURE)
print(f'LLM ready: {LLM_MODEL} (temperature={LLM_TEMPERATURE}) — OPENAI_API_KEY set: {OPENAI_KEY_SET}')

if not OPENAI_KEY_SET:
    print('⚠️  OPENAI_API_KEY 가 비어있어 LLM 호출은 fallback 경로로 빠집니다. .env 에 키를 추가하세요.')

## 1. contracts/ — 데이터 계약 스키마

In [ ]:
class RouteDecision(BaseModel):
    next_node: str
    reason: str
    stop: bool = False

class GateReport(BaseModel):
    gate_name: str
    status: str
    route_hint: Optional[str] = None
    reason: str = ''
    details: dict = Field(default_factory=dict)

class InputFlags(BaseModel):
    possible_manufacturing_query: bool = False
    possible_prediction_query: bool = False
    possible_safety_query: bool = False
    possible_prompt_injection: bool = False
    contains_sensor_values: bool = False
    blocked_by_raw_input: bool = False

In [ ]:
class PredictionResult(BaseModel):
    status: str
    available_features: list[str] = []
    missing_features: list[str] = []
    full_prediction_available: bool = False
    partial_risks: list[dict] = []
    ml_prediction: Optional[dict] = None
    summary: str = ''

class EvidenceBundle(BaseModel):
    retrieval_profile: str = ''
    queries: list[str] = []
    documents: list[dict] = []
    citations: list[dict] = []
    evidence_summary: str = ''
    sufficient: bool = True

class SafetyDecision(BaseModel):
    risk_level: str = 'unknown'
    blocked: bool = False
    forbidden_actions: list[str] = []
    required_safety_notes: list[str] = []
    summary: str = ''

class FinalAnswer(BaseModel):
    answer: str
    citations: list[dict] = []
    warnings: list[str] = []
    missing_inputs: list[str] = []

In [ ]:
class ConversationTurn(BaseModel):
    role: str
    content: str
    created_at: str

class MachineValue(BaseModel):
    name: str
    value: float | str
    unit: Optional[str] = None
    source: str
    is_current: bool
    is_stale: bool = False

class ContextPacket(BaseModel):
    current_question: str
    recent_turns_summary: str = ''
    selected_machine_values: dict[str, MachineValue] = {}
    previous_prediction_summary: Optional[str] = None
    previous_safety_summary: Optional[str] = None
    user_constraints: dict = {}
    context_warnings: list[str] = []

class AgentContextPacket(BaseModel):
    agent_name: str
    current_question: str
    selected_context: dict
    prior_results: dict = {}

In [ ]:
class ManufacturingState(TypedDict, total=False):
    request_id: str
    session_id: str
    user_message: str

    input_flags: dict
    route: dict
    intent: Optional[str]

    context_packet: dict
    agent_contexts: dict[str, dict]

    prediction_result: dict
    evidence_bundle: dict
    safety_decision: dict
    final_answer: dict

    gate_reports: list[dict]
    retry_counts: dict[str, int]
    run_trace: list[dict]

## 2. memory/ — 장기 메모리 (SQLite)

In [ ]:
class SqliteLongTermMemory:
    def __init__(self, db_path: str):
        self.db_path = db_path
        self._init_schema()

    def _conn(self):
        return sqlite3.connect(self.db_path)

    def _init_schema(self):
        with self._conn() as c:
            c.executescript(
                '''
                CREATE TABLE IF NOT EXISTS conversation_turns (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    session_id TEXT NOT NULL,
                    role TEXT NOT NULL,
                    content TEXT NOT NULL,
                    created_at TEXT NOT NULL
                );
                CREATE INDEX IF NOT EXISTS idx_turns_session ON conversation_turns(session_id);

                CREATE TABLE IF NOT EXISTS turn_summaries (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    session_id TEXT NOT NULL,
                    request_id TEXT NOT NULL,
                    machine_values_json TEXT,
                    prediction_summary TEXT,
                    safety_summary TEXT,
                    missing_features_json TEXT,
                    created_at TEXT NOT NULL
                );
                CREATE INDEX IF NOT EXISTS idx_summary_session ON turn_summaries(session_id);

                CREATE TABLE IF NOT EXISTS run_traces (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    request_id TEXT NOT NULL,
                    session_id TEXT NOT NULL,
                    gate_reports_json TEXT,
                    retry_counts_json TEXT,
                    trace_json TEXT,
                    created_at TEXT NOT NULL
                );
                '''
            )

    def append_turn(self, session_id, role, content):
        with self._conn() as c:
            c.execute('INSERT INTO conversation_turns(session_id, role, content, created_at) VALUES (?,?,?,?)',
                      (session_id, role, content, datetime.now(timezone.utc).isoformat()))

    def recent_turns(self, session_id, limit=10):
        with self._conn() as c:
            rows = c.execute('SELECT role, content, created_at FROM conversation_turns WHERE session_id=? ORDER BY id DESC LIMIT ?',
                             (session_id, limit)).fetchall()
        return [{'role': r[0], 'content': r[1], 'created_at': r[2]} for r in reversed(rows)]

    def save_turn_summary(self, session_id, request_id, *, machine_values, prediction_summary,
                          safety_summary, missing_features):
        with self._conn() as c:
            c.execute('''INSERT INTO turn_summaries(session_id, request_id, machine_values_json,
                         prediction_summary, safety_summary, missing_features_json, created_at)
                         VALUES (?,?,?,?,?,?,?)''',
                      (session_id, request_id, json.dumps(machine_values, ensure_ascii=False),
                       prediction_summary, safety_summary,
                       json.dumps(missing_features, ensure_ascii=False),
                       datetime.now(timezone.utc).isoformat()))

    def latest_summary(self, session_id):
        with self._conn() as c:
            row = c.execute('''SELECT machine_values_json, prediction_summary, safety_summary, missing_features_json
                              FROM turn_summaries WHERE session_id=? ORDER BY id DESC LIMIT 1''',
                            (session_id,)).fetchone()
        if not row:
            return None
        return {
            'machine_values': json.loads(row[0] or '{}'),
            'prediction_summary': row[1] or '',
            'safety_summary': row[2] or '',
            'missing_features': json.loads(row[3] or '[]'),
        }

    def save_run_trace(self, request_id, session_id, *, gate_reports, retry_counts, trace):
        with self._conn() as c:
            c.execute('''INSERT INTO run_traces(request_id, session_id, gate_reports_json,
                         retry_counts_json, trace_json, created_at) VALUES (?,?,?,?,?,?)''',
                      (request_id, session_id,
                       json.dumps(gate_reports, ensure_ascii=False),
                       json.dumps(retry_counts, ensure_ascii=False),
                       json.dumps(trace, ensure_ascii=False),
                       datetime.now(timezone.utc).isoformat()))


LONG_TERM = SqliteLongTermMemory(LONGTERM_DB)
print('Long-term memory ready')

## 3. context/ — Context Engineering 파이프라인

In [ ]:
CANONICAL_FEATURES = ['type', 'air_temperature', 'process_temperature',
                     'rotational_speed', 'torque', 'tool_wear']

INJECTION_PATTERNS = [
    r'안전\s*경고는?\s*하지\s*말',
    r'무시(하고|해)',
    r'ignore\s+(previous|all)\s+instructions',
    r'시스템\s*프롬프트',
]

SENSOR_PATTERNS = {
    'air_temperature': r'(?:air[_\s]?temp(?:erature)?|공기\s*온도)\s*[:=]?\s*([0-9]+(?:\.[0-9]+)?)',
    'process_temperature': r'(?:process[_\s]?temp(?:erature)?|공정\s*온도)\s*[:=]?\s*([0-9]+(?:\.[0-9]+)?)',
    'rotational_speed': r'(?:rpm|회전수|rotational[_\s]?speed)\s*[:=]?\s*([0-9]+(?:\.[0-9]+)?)',
    'torque': r'(?:torque|토크)\s*[:=]?\s*([0-9]+(?:\.[0-9]+)?)',
    'tool_wear': r'(?:tool[_\s]?wear|공구\s*마모)\s*[:=]?\s*([0-9]+(?:\.[0-9]+)?)',
    'type': r'(?:type|품질등급)\s*[:=]?\s*([LMH])',
}

def extract_machine_values(text: str) -> dict[str, Any]:
    out: dict[str, Any] = {}
    for name, pat in SENSOR_PATTERNS.items():
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            val = m.group(1)
            try:
                out[name] = float(val) if name != 'type' else val.upper()
            except ValueError:
                out[name] = val
    return out

def sanitize_injection(text: str) -> str:
    for pat in INJECTION_PATTERNS:
        text = re.sub(pat, '[정책위반문구제거]', text, flags=re.IGNORECASE)
    return text

In [ ]:
def select_and_normalize(current_msg: str, prior_summary: Optional[dict]) -> tuple[dict, list[str]]:
    warnings: list[str] = []
    current = extract_machine_values(current_msg)

    merged: dict[str, MachineValue] = {}
    for k, v in current.items():
        merged[k] = MachineValue(name=k, value=v, source='current', is_current=True)

    if prior_summary and prior_summary.get('machine_values'):
        for k, v in prior_summary['machine_values'].items():
            if k in merged:
                continue
            merged[k] = MachineValue(name=k, value=v, source='previous_turn',
                                     is_current=False, is_stale=True)
        if current and prior_summary['machine_values']:
            overlap = set(current) & set(prior_summary['machine_values'])
            for k in overlap:
                if current[k] != prior_summary['machine_values'].get(k):
                    warnings.append(f'feature {k}: 현재값={current[k]} / 이전값={prior_summary["machine_values"][k]} (현재값 채택)')

    return {k: v.model_dump() for k, v in merged.items()}, warnings

In [ ]:
def build_recent_summary(turns: list[dict], budget_chars: int = 600) -> str:
    if not turns:
        return ''
    pieces = []
    for t in turns[-6:]:
        snippet = sanitize_injection(t['content'])[:200]
        pieces.append(f"[{t['role']}] {snippet}")
    return '\n'.join(pieces)[-budget_chars:]

def pack_agent_contexts(packet: ContextPacket) -> dict[str, AgentContextPacket]:
    base_q = packet.current_question
    machine_values = packet.selected_machine_values
    return {
        'prediction': AgentContextPacket(
            agent_name='prediction', current_question=base_q,
            selected_context={
                'machine_values': {k: v.model_dump() for k, v in machine_values.items()},
                'recent_turns_summary': packet.recent_turns_summary,
                'context_warnings': packet.context_warnings,
            }),
        'evidence': AgentContextPacket(
            agent_name='evidence', current_question=base_q,
            selected_context={
                'recent_turns_summary': packet.recent_turns_summary,
                'previous_prediction_summary': packet.previous_prediction_summary,
            }),
        'safety': AgentContextPacket(
            agent_name='safety', current_question=base_q,
            selected_context={
                'previous_safety_summary': packet.previous_safety_summary,
                'recent_turns_summary': packet.recent_turns_summary,
            }),
        'final_answer': AgentContextPacket(
            agent_name='final_answer', current_question=base_q,
            selected_context={
                'recent_turns_summary': packet.recent_turns_summary,
                'context_warnings': packet.context_warnings,
            }),
    }

In [ ]:
def context_manager_node(state: ManufacturingState) -> dict:
    session_id = state['session_id']
    user_msg = state['user_message']
    sanitized_msg = sanitize_injection(user_msg)
    turns = LONG_TERM.recent_turns(session_id, limit=10)
    prior = LONG_TERM.latest_summary(session_id)
    selected, warnings = select_and_normalize(sanitized_msg, prior)
    recent_summary = build_recent_summary(turns)

    packet = ContextPacket(
        current_question=sanitized_msg,
        recent_turns_summary=recent_summary,
        selected_machine_values={k: MachineValue(**v) for k, v in selected.items()},
        previous_prediction_summary=(prior or {}).get('prediction_summary'),
        previous_safety_summary=(prior or {}).get('safety_summary'),
        context_warnings=warnings,
    )
    agent_ctxs = pack_agent_contexts(packet)
    return to_jsonable({
        'context_packet': packet.model_dump(),
        'agent_contexts': {k: v.model_dump() for k, v in agent_ctxs.items()},
    })

## 4. services/ — Prediction(RandomForest) / RAG / Safety / Citation

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

AI4I_CSV = DATA_DIR / 'ai4i2020.csv'

FEATURE_COLS = [
    'Air temperature [K]', 'Process temperature [K]',
    'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
]
NUMERIC_KEYS = ['air_temperature', 'process_temperature', 'rotational_speed', 'torque', 'tool_wear']
TYPE_LEVELS = ['L', 'M', 'H']

REQUIRED_BY_FAULT = {
    'HDF': ['air_temperature', 'process_temperature', 'rotational_speed'],
    'PWF': ['rotational_speed', 'torque'],
    'OSF': ['tool_wear', 'torque', 'type'],
    'TWF': ['tool_wear'],
}


class PredictionService:
    """RandomForest + Type one-hot. 클래스 불균형은 class_weight='balanced'."""

    def __init__(self, csv_path: Path):
        self.df = pd.read_csv(csv_path)
        X_num = self.df[FEATURE_COLS].values
        type_oh = pd.get_dummies(self.df['Type'])[TYPE_LEVELS].astype(float).values
        X = np.hstack([X_num, type_oh])
        y = self.df['Machine failure'].values

        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                                  stratify=y, random_state=42)
        self.model = RandomForestClassifier(
            n_estimators=300, max_depth=None,
            class_weight='balanced', random_state=42, n_jobs=-1,
        )
        self.model.fit(X_tr, y_tr)
        y_pred = self.model.predict(X_te)
        self.eval_report = classification_report(y_te, y_pred, digits=3, output_dict=False)
        self.feature_names = NUMERIC_KEYS + [f'type_{lvl}' for lvl in TYPE_LEVELS]

    @staticmethod
    def _to_input_row(values: dict) -> Optional[list[float]]:
        if any(k not in values for k in NUMERIC_KEYS):
            return None
        if 'type' not in values:
            return None
        row = [float(values[k]) for k in NUMERIC_KEYS]
        tp = str(values['type']).upper()
        row += [1.0 if tp == lvl else 0.0 for lvl in TYPE_LEVELS]
        return row

    def ml_predict(self, values: dict) -> Optional[dict]:
        row = self._to_input_row(values)
        if row is None:
            return None
        X = np.array([row])
        proba = float(self.model.predict_proba(X)[0][1])
        # v0.2 fix: numpy.float64 → Python float 명시 변환
        importances = {n: float(v) for n, v in zip(self.feature_names, self.model.feature_importances_)}
        top_feats = sorted(importances.items(), key=lambda x: -x[1])[:3]
        return {
            'p_failure': proba,
            'label': int(proba >= 0.5),
            'model': f'RandomForest(n=300, balanced)',
            'top_feature_importances': [[n, float(v)] for n, v in top_feats],
        }

    @staticmethod
    def rule_based_partial(values: dict) -> list[dict]:
        out = []
        if all(k in values for k in ['air_temperature', 'process_temperature', 'rotational_speed']):
            dT = float(values['process_temperature']) - float(values['air_temperature'])
            rpm = float(values['rotational_speed'])
            triggered = dT < 8.6 and rpm < 1380
            out.append({'fault': 'HDF', 'triggered': bool(triggered),
                        'evidence': f'ΔT={dT:.2f}K, rpm={rpm}'})
        if all(k in values for k in ['rotational_speed', 'torque']):
            rpm = float(values['rotational_speed'])
            torque = float(values['torque'])
            power = torque * rpm * (2 * 3.14159 / 60)
            triggered = power < 3500 or power > 9000
            out.append({'fault': 'PWF', 'triggered': bool(triggered),
                        'evidence': f'Power≈{power:.1f}W'})
        if all(k in values for k in ['tool_wear', 'torque', 'type']):
            wear_torque = float(values['tool_wear']) * float(values['torque'])
            thresh = {'H': 11000, 'M': 12000, 'L': 13000}.get(str(values['type']).upper())
            triggered = thresh is not None and wear_torque > thresh
            out.append({'fault': 'OSF', 'triggered': bool(triggered),
                        'evidence': f'wear*torque={wear_torque:.0f}, thresh={thresh}'})
        if 'tool_wear' in values:
            tw = float(values['tool_wear'])
            triggered = 200 <= tw <= 240
            out.append({'fault': 'TWF', 'triggered': bool(triggered),
                        'evidence': f'tool_wear={tw}min'})
        return out


PREDICTION_SERVICE = PredictionService(AI4I_CSV)
print('Prediction service: RandomForest trained on', len(PREDICTION_SERVICE.df), 'rows')
print(PREDICTION_SERVICE.eval_report)

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

RETRIEVAL_PROFILES = {
    'prediction_plus_rag': '예측 결과를 문서 근거로 설명',
    'troubleshooting_rag': '장비 이상 증상/트러블슈팅',
    'safety_rag': 'LOTO, 정비 안전, 위험 작업',
    'concept_explanation': '제조 개념 설명',
    'fallback_broad': '근거 부족 시 확장 검색',
}

SEED_DOCS = [
    {'id': 'doc-hdf-001', 'title': 'Heat Dissipation Failure 가이드',
     'source': 'milling-machine-manual-v2.1.pdf', 'profile': 'prediction_plus_rag',
     'text': 'HDF는 공정-공기 온도차가 8.6K 미만이고 회전수가 1380rpm 미만일 때 방열 능력이 부족해 발생한다. 대응: 냉각수 유량 점검, 회전수 상향, 공구 점검.'},
    {'id': 'doc-pwf-001', 'title': 'Power Failure 진단',
     'source': 'milling-machine-manual-v2.1.pdf', 'profile': 'prediction_plus_rag',
     'text': 'PWF는 토크와 회전수로 산출한 기계적 출력이 3500W 미만이거나 9000W 초과 시 발생한다. 모터 부하, 베어링 상태를 점검한다.'},
    {'id': 'doc-osf-001', 'title': 'Overstrain Failure 임계값',
     'source': 'milling-machine-manual-v2.1.pdf', 'profile': 'prediction_plus_rag',
     'text': 'OSF는 공구마모(min)와 토크(Nm)의 곱이 품질등급별 임계값(L:13000, M:12000, H:11000)을 초과하면 발생한다.'},
    {'id': 'doc-twf-001', 'title': 'Tool Wear Failure',
     'source': 'milling-machine-manual-v2.1.pdf', 'profile': 'troubleshooting_rag',
     'text': 'TWF는 공구 누적 사용시간이 200~240분 구간에서 무작위로 발생한다. 200분 이전 예방교체가 권장된다.'},
    {'id': 'doc-loto-001', 'title': 'LOTO 절차',
     'source': 'plant-safety-policy-2025.pdf', 'profile': 'safety_rag',
     'text': '정비 작업 전 전원 차단, 잔류 에너지 방출, 잠금장치 및 태그 부착이 필수다. 운전 중 정비/조정 작업은 금지된다.'},
    {'id': 'doc-safety-002', 'title': '위험 운전 지속 금지',
     'source': 'plant-safety-policy-2025.pdf', 'profile': 'safety_rag',
     'text': '예측 위험 신호가 발생한 설비를 정지 없이 지속 운전해서는 안 된다. 라인 매니저 승인과 추가 점검이 선행되어야 한다.'},
]


class RagService:
    def __init__(self, chroma_path: str, collection: str = 'manufacturing_docs'):
        self.client = chromadb.PersistentClient(path=chroma_path)
        self.embed_fn = embedding_functions.DefaultEmbeddingFunction()
        self.collection = self.client.get_or_create_collection(
            name=collection, embedding_function=self.embed_fn)
        if self.collection.count() == 0:
            self._seed()

    def _seed(self):
        self.collection.add(
            ids=[d['id'] for d in SEED_DOCS],
            documents=[d['text'] for d in SEED_DOCS],
            metadatas=[{'title': d['title'], 'source': d['source'], 'profile': d['profile']}
                       for d in SEED_DOCS])

    def search(self, query: str, *, profile: str, n: int = 3) -> list[dict]:
        where = {'profile': profile} if profile == 'safety_rag' else None
        res = self.collection.query(query_texts=[query], n_results=n, where=where)
        out = []
        for i, doc in enumerate(res['documents'][0]):
            meta = res['metadatas'][0][i]
            # v0.2 fix: distance도 Python float로 변환 (Chroma가 numpy로 반환할 수 있음)
            dist = res['distances'][0][i] if res.get('distances') else None
            out.append({
                'doc_id': res['ids'][0][i], 'text': doc,
                'title': meta.get('title'), 'source': meta.get('source'),
                'profile': meta.get('profile'),
                'distance': float(dist) if dist is not None else None,
            })
        return out


RAG = RagService(CHROMA_PATH)
print('Chroma collection size:', RAG.collection.count())

In [ ]:
FORBIDDEN_REQUEST_PATTERNS = [
    (r'정지\s*없이|운전\s*지속|그냥\s*계속\s*돌려', '경고 발생 설비의 무정지 지속운전 요청'),
    (r'안전\s*경고\s*(없이|생략)', '안전 경고 생략 요청'),
    (r'loto\s*(없이|생략)', 'LOTO 절차 생략 요청'),
]

def regex_forbidden_actions(question: str) -> list[str]:
    out = []
    for pat, label in FORBIDDEN_REQUEST_PATTERNS:
        if re.search(pat, question.lower()):
            out.append(label)
    return out


class CitationService:
    @staticmethod
    def normalize(documents: list[dict]) -> list[dict]:
        return [{
            'cite_id': d['doc_id'], 'title': d.get('title'),
            'source': d.get('source'), 'snippet': d['text'][:160],
        } for d in documents]


CITATIONS = CitationService()

## 5. LLM 헬퍼 — 구조화 출력 스키마

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

class QueryPlan(BaseModel):
    retrieval_profile: str = Field(description='prediction_plus_rag/troubleshooting_rag/safety_rag/concept_explanation/fallback_broad 중 하나')
    queries: list[str] = Field(description='검색용 한국어 쿼리 2~4개. 원 질문 + 예측 결과를 반영')

class EvidenceGrade(BaseModel):
    sufficient: bool = Field(description='이 문서들이 사용자 질문에 답하기 충분한가')
    summary: str = Field(description='검색된 근거 요약 (3~5문장)')
    used_cite_ids: list[str] = Field(description='답변에 인용할 doc_id 목록')

class LLMSafetyDecision(BaseModel):
    risk_level: str = Field(description='low/medium/high/critical 중 하나')
    blocked: bool = Field(description='안전 정책상 응답 차단 여부')
    forbidden_actions: list[str] = Field(description='사용자가 요구한 금지 행동 목록')
    required_safety_notes: list[str] = Field(description='반드시 포함되어야 할 안전 주의사항')
    summary: str = Field(description='판단 근거 요약 (1~2문장)')

class LLMFinalAnswer(BaseModel):
    answer_markdown: str = Field(description='한국어 마크다운 답변. 예측/근거/안전을 통합해 명확하고 단정 표현을 피한다.')
    cited_ids: list[str] = Field(description='답변에서 인용한 doc_id 목록 (citations와 매칭)')


def safe_llm_invoke(structured_llm, prompt_msgs, *, fallback):
    try:
        return structured_llm.invoke(prompt_msgs)
    except Exception as e:
        print(f'[LLM fallback] {type(e).__name__}: {e}')
        return fallback

## 6. agents/ — Prediction(ML) / Evidence(LLM+RAG) / Safety(LLM+규칙)

**v0.2**: 모든 agent 노드는 반환 직전에 `to_jsonable()`을 적용해 numpy 타입을 제거한다.

In [ ]:
def prediction_agent_node(state: ManufacturingState) -> dict:
    ctx = state['agent_contexts']['prediction']
    values_dump = ctx['selected_context']['machine_values']
    values = {k: v['value'] for k, v in values_dump.items()}

    available = sorted(values.keys())
    needed = NUMERIC_KEYS + ['type']
    missing = [k for k in needed if k not in values]

    partial = PREDICTION_SERVICE.rule_based_partial(values)
    ml = PREDICTION_SERVICE.ml_predict(values) if not missing else None

    status = 'OK' if ml else ('PARTIAL' if partial else 'SKIPPED')
    lines = []
    if ml:
        lines.append(f"ML(RF) P(failure)={ml['p_failure']:.2%}")
    for p in partial:
        lines.append(f"{p['fault']}: {'⚠️ TRIGGER' if p['triggered'] else 'ok'} ({p['evidence']})")
    if missing:
        lines.append(f'누락 feature: {missing}')

    result = PredictionResult(
        status=status, available_features=available, missing_features=missing,
        full_prediction_available=ml is not None,
        partial_risks=partial, ml_prediction=ml,
        summary=' / '.join(lines) or '입력값 부족으로 판단 불가',
    )
    # v0.2 fix: numpy 잔여 타입 제거
    return to_jsonable({'prediction_result': result.model_dump()})

In [ ]:
EVIDENCE_PLAN_PROMPT = ChatPromptTemplate.from_messages([
    ('system', '당신은 제조 RAG 검색 플래너다. 사용자 질문과 예측 결과를 보고 '
               '적절한 retrieval_profile과 한국어 검색 쿼리 2~4개를 생성한다. '
               'retrieval_profile은 반드시 다음 중 하나: prediction_plus_rag, troubleshooting_rag, safety_rag, concept_explanation, fallback_broad.'),
    ('human', '질문: {question}\n예측 요약: {prediction_summary}\npartial risks: {partial_risks}'),
])

EVIDENCE_GRADE_PROMPT = ChatPromptTemplate.from_messages([
    ('system', '검색된 문서들이 사용자 질문에 충분한 근거를 제공하는지 평가한다. '
               '사용 가능한 doc_id 목록만 used_cite_ids에 포함한다 (없으면 빈 리스트).'),
    ('human', '질문: {question}\n\n검색 문서들:\n{docs}'),
])


def evidence_agent_node(state: ManufacturingState) -> dict:
    ctx = state['agent_contexts']['evidence']
    question = ctx['current_question']
    pr = state.get('prediction_result') or {}

    plan_llm = LLM.with_structured_output(QueryPlan)
    plan = safe_llm_invoke(
        plan_llm,
        EVIDENCE_PLAN_PROMPT.format_messages(
            question=question,
            prediction_summary=pr.get('summary', '없음'),
            partial_risks=json.dumps(pr.get('partial_risks', []), ensure_ascii=False),
        ),
        fallback=QueryPlan(retrieval_profile='fallback_broad', queries=[question]),
    )

    docs: list[dict] = []
    seen = set()
    for q in plan.queries:
        for d in RAG.search(q, profile=plan.retrieval_profile, n=3):
            if d['doc_id'] in seen:
                continue
            seen.add(d['doc_id'])
            docs.append(d)

    docs_text = '\n'.join(
        f"- [{d['doc_id']}] {d['title']}: {d['text'][:200]}" for d in docs)
    grade_llm = LLM.with_structured_output(EvidenceGrade)
    grade = safe_llm_invoke(
        grade_llm,
        EVIDENCE_GRADE_PROMPT.format_messages(question=question, docs=docs_text or '없음'),
        fallback=EvidenceGrade(
            sufficient=bool(docs),
            summary=f'{len(docs)}건 검색됨 (LLM grading 불가)',
            used_cite_ids=[d['doc_id'] for d in docs],
        ),
    )

    valid_ids = {d['doc_id'] for d in docs}
    used_docs = [d for d in docs if d['doc_id'] in set(grade.used_cite_ids) & valid_ids]
    citations = CITATIONS.normalize(used_docs or docs)

    bundle = EvidenceBundle(
        retrieval_profile=plan.retrieval_profile,
        queries=plan.queries, documents=docs,
        citations=citations,
        evidence_summary=grade.summary,
        sufficient=grade.sufficient,
    )
    return to_jsonable({'evidence_bundle': bundle.model_dump()})

In [ ]:
SAFETY_PROMPT = ChatPromptTemplate.from_messages([
    ('system', '당신은 제조 현장 안전 정책 판단자다. 사용자 질문, 예측 결과, 근거 문서를 보고 '
               '한국 산업안전 기준(LOTO/정비/운전 지속)에 따라 risk_level과 차단 여부를 판단한다. '
               '운전 중 정비 요청, 안전 경고 생략 요청, 위험 신호 발생 설비의 무정지 지속운전 요청은 critical로 분류하고 blocked=True. '
               'LOTO와 관리자 승인 관련 노트는 항상 포함.'),
    ('human', '질문: {question}\n예측 요약: {prediction_summary}\n근거 요약: {evidence_summary}\nregex로 감지된 금지요청: {regex_hits}'),
])


def safety_agent_node(state: ManufacturingState) -> dict:
    ctx = state['agent_contexts']['safety']
    question = ctx['current_question']
    pr = state.get('prediction_result') or {}
    eb = state.get('evidence_bundle') or {}

    regex_hits = regex_forbidden_actions(question)

    safety_llm = LLM.with_structured_output(LLMSafetyDecision)
    llm_decision = safe_llm_invoke(
        safety_llm,
        SAFETY_PROMPT.format_messages(
            question=question,
            prediction_summary=pr.get('summary', '없음'),
            evidence_summary=eb.get('evidence_summary', '없음'),
            regex_hits=regex_hits or '없음',
        ),
        fallback=LLMSafetyDecision(
            risk_level='critical' if regex_hits else 'medium',
            blocked=bool(regex_hits),
            forbidden_actions=regex_hits,
            required_safety_notes=[
                '정비/점검 작업 시 LOTO 절차 적용',
                '예측 위험 신호 발생 설비는 즉시 정지',
            ],
            summary='LLM 호출 실패 — 보수적 fallback',
        ),
    )

    if regex_hits and not llm_decision.blocked:
        llm_decision.blocked = True
        llm_decision.risk_level = 'critical'
        for h in regex_hits:
            if h not in llm_decision.forbidden_actions:
                llm_decision.forbidden_actions.append(h)

    decision = SafetyDecision(
        risk_level=llm_decision.risk_level,
        blocked=llm_decision.blocked,
        forbidden_actions=llm_decision.forbidden_actions,
        required_safety_notes=llm_decision.required_safety_notes,
        summary=llm_decision.summary,
    )
    return to_jsonable({'safety_decision': decision.model_dump()})

## 7. gates/ — 검증 게이트

In [ ]:
def _append_report(state: ManufacturingState, report: GateReport) -> list[dict]:
    reports = list(state.get('gate_reports') or [])
    reports.append(report.model_dump())
    return reports


def input_gate_node(state: ManufacturingState) -> dict:
    msg = state['user_message'] or ''
    flags = InputFlags(
        possible_manufacturing_query=any(k in msg.lower() for k in ['설비', '공정', 'rpm', '토크', '온도', '마모', '고장']),
        possible_prediction_query=any(k in msg for k in ['예측', '위험', '고장날', '확률']),
        possible_safety_query=any(k in msg for k in ['안전', '정비', 'LOTO', '운전 지속']),
        possible_prompt_injection=any(re.search(p, msg, re.IGNORECASE) for p in INJECTION_PATTERNS),
        contains_sensor_values=bool(extract_machine_values(msg)),
        blocked_by_raw_input=not msg.strip(),
    )
    status = 'BLOCK' if flags.blocked_by_raw_input else 'PASS'
    report = GateReport(gate_name='input_gate', status=status,
                        reason='ok' if status == 'PASS' else '빈 입력')
    return to_jsonable({'input_flags': flags.model_dump(), 'gate_reports': _append_report(state, report)})


def prediction_gate_node(state: ManufacturingState) -> dict:
    pr = state.get('prediction_result') or {}
    if not pr:
        status = 'FAIL'
    elif pr.get('full_prediction_available'):
        status = 'PASS'
    elif pr.get('partial_risks'):
        status = 'PASS_WITH_PARTIAL_RESULT'
    else:
        status = 'ASK_MISSING_INPUT'
    report = GateReport(gate_name='prediction_gate', status=status,
                        reason=pr.get('summary', ''),
                        details={'missing': pr.get('missing_features', [])})
    return to_jsonable({'gate_reports': _append_report(state, report)})


def evidence_gate_node(state: ManufacturingState) -> dict:
    eb = state.get('evidence_bundle') or {}
    if not eb.get('documents'):
        status = 'INSUFFICIENT_EVIDENCE'
    elif not eb.get('sufficient', True):
        status = 'RETRY_WITH_EXPANDED_QUERY'
    elif not eb.get('citations'):
        status = 'RETRY_WITH_EXPANDED_QUERY'
    else:
        status = 'PASS'
    report = GateReport(gate_name='evidence_gate', status=status,
                        reason=eb.get('evidence_summary', ''))
    return to_jsonable({'gate_reports': _append_report(state, report)})


def safety_gate_node(state: ManufacturingState) -> dict:
    sd = state.get('safety_decision') or {}
    if not sd:
        status = 'REQUIRE_SAFETY_DECISION'
    elif sd.get('blocked'):
        status = 'BLOCK'
    elif sd.get('risk_level') == 'high':
        status = 'REWRITE_SAFE'
    else:
        status = 'PASS'
    report = GateReport(gate_name='safety_gate', status=status, reason=sd.get('summary', ''))
    return to_jsonable({'gate_reports': _append_report(state, report)})


def output_gate_node(state: ManufacturingState) -> dict:
    fa = state.get('final_answer') or {}
    answer = fa.get('answer', '')
    sd = state.get('safety_decision') or {}
    eb = state.get('evidence_bundle') or {}
    issues = []
    if not answer:
        issues.append('빈 답변')
    if eb.get('citations') and not fa.get('citations'):
        issues.append('citation 누락')
    if sd.get('blocked') and ('차단' not in answer and 'BLOCK' not in answer.upper()):
        issues.append('SafetyDecision blocked 미반영')
    status = 'PASS' if not issues else 'REWRITE'
    report = GateReport(gate_name='output_gate', status=status, reason=', '.join(issues) or 'ok')
    return to_jsonable({'gate_reports': _append_report(state, report)})

## 8. nodes/ — FinalAnswer(LLM) + MemoryWriter

In [ ]:
FINAL_ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ('system', '당신은 제조 AI 어시스턴트다. 사용자에게 한국어 마크다운으로 답한다. '
               '규칙: (1) 안전 차단(blocked=True)이면 정중히 거부하고 안전 노트만 제시. '
               '(2) 누락 feature가 있으면 단정하지 말고 한계를 명시. '
               '(3) 답변에서 인용한 doc_id는 cited_ids에 반드시 명시. '
               '(4) 모든 위험 신호와 안전 주의사항을 빠짐없이 포함한다. '
               '(5) 250~500자 사이 간결한 답변.'),
    ('human', '질문: {question}\n\n예측: {prediction}\n\n근거 문서:\n{evidence}\n\n안전 판단: {safety}\n\nContext 경고: {warnings}'),
])


def final_answer_node(state: ManufacturingState) -> dict:
    pr = state.get('prediction_result') or {}
    eb = state.get('evidence_bundle') or {}
    sd = state.get('safety_decision') or {}
    ctx = state.get('context_packet') or {}

    evidence_text = '\n'.join(
        f"- [{c['cite_id']}] {c['title']} ({c['source']}): {c['snippet']}"
        for c in eb.get('citations', [])
    ) or '없음'

    answer_llm = LLM.with_structured_output(LLMFinalAnswer)
    llm_out = safe_llm_invoke(
        answer_llm,
        FINAL_ANSWER_PROMPT.format_messages(
            question=ctx.get('current_question', state.get('user_message', '')),
            prediction=json.dumps({
                'status': pr.get('status'),
                'summary': pr.get('summary'),
                'missing': pr.get('missing_features'),
                'ml': pr.get('ml_prediction'),
                'partial': pr.get('partial_risks'),
            }, ensure_ascii=False),
            evidence=evidence_text,
            safety=json.dumps(sd, ensure_ascii=False),
            warnings=ctx.get('context_warnings', []),
        ),
        fallback=LLMFinalAnswer(
            answer_markdown=f"## 분석 (LLM fallback)\n- 상태: {pr.get('status')}\n- 요약: {pr.get('summary')}\n- 안전: {sd.get('summary')}",
            cited_ids=[c['cite_id'] for c in eb.get('citations', [])],
        ),
    )

    valid_cites = {c['cite_id'] for c in eb.get('citations', [])}
    final_citations = [c for c in eb.get('citations', []) if c['cite_id'] in set(llm_out.cited_ids) & valid_cites]
    if not final_citations:
        final_citations = eb.get('citations', [])

    final = FinalAnswer(
        answer=llm_out.answer_markdown,
        citations=final_citations,
        warnings=ctx.get('context_warnings', []),
        missing_inputs=pr.get('missing_features', []),
    )
    return to_jsonable({'final_answer': final.model_dump()})


def memory_writer_node(state: ManufacturingState) -> dict:
    session_id = state['session_id']
    request_id = state['request_id']

    LONG_TERM.append_turn(session_id, 'user', state['user_message'])
    LONG_TERM.append_turn(session_id, 'assistant', (state.get('final_answer') or {}).get('answer', ''))

    pr = state.get('prediction_result') or {}
    sd = state.get('safety_decision') or {}
    ctx = state.get('context_packet') or {}

    machine_values = {
        k: v['value'] for k, v in (ctx.get('selected_machine_values') or {}).items()
        if v.get('is_current')
    }
    LONG_TERM.save_turn_summary(
        session_id, request_id,
        machine_values=to_jsonable(machine_values),
        prediction_summary=pr.get('summary', ''),
        safety_summary=sd.get('summary', ''),
        missing_features=pr.get('missing_features', []),
    )
    LONG_TERM.save_run_trace(
        request_id, session_id,
        gate_reports=to_jsonable(state.get('gate_reports') or []),
        retry_counts=to_jsonable(state.get('retry_counts') or {}),
        trace=to_jsonable(state.get('run_trace') or []),
    )
    return {}

## 9. graph/ — Supervisor + Route Policy + Graph 빌드

In [ ]:
def _latest_report(state: ManufacturingState, gate_name: str) -> Optional[dict]:
    for r in reversed(state.get('gate_reports') or []):
        if r['gate_name'] == gate_name:
            return r
    return None


def supervisor_node(state: ManufacturingState) -> dict:
    pr = state.get('prediction_result')
    eb = state.get('evidence_bundle')
    sd = state.get('safety_decision')
    input_report = _latest_report(state, 'input_gate') or {}

    if input_report.get('status') == 'BLOCK':
        decision = RouteDecision(next_node='final_answer', reason='input blocked')
    elif pr is None:
        decision = RouteDecision(next_node='prediction_agent', reason='prediction required')
    elif eb is None:
        decision = RouteDecision(next_node='evidence_agent', reason='evidence required')
    elif sd is None:
        decision = RouteDecision(next_node='safety_agent', reason='safety decision required')
    else:
        decision = RouteDecision(next_node='final_answer', reason='all results ready')
    return {'route': decision.model_dump()}


def supervisor_router(state: ManufacturingState) -> str:
    return (state.get('route') or {}).get('next_node', 'final_answer')

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

try:
    from langgraph.checkpoint.sqlite import SqliteSaver
    _HAS_SQLITE_SAVER = True
except ImportError:
    _HAS_SQLITE_SAVER = False


def build_graph(checkpointer=None):
    g = StateGraph(ManufacturingState)
    g.add_node('input_gate', input_gate_node)
    g.add_node('context_manager', context_manager_node)
    g.add_node('supervisor', supervisor_node)
    g.add_node('prediction_agent', prediction_agent_node)
    g.add_node('prediction_gate', prediction_gate_node)
    g.add_node('evidence_agent', evidence_agent_node)
    g.add_node('evidence_gate', evidence_gate_node)
    g.add_node('safety_agent', safety_agent_node)
    g.add_node('safety_gate', safety_gate_node)
    g.add_node('final_answer', final_answer_node)
    g.add_node('output_gate', output_gate_node)
    g.add_node('memory_writer', memory_writer_node)

    g.add_edge(START, 'input_gate')
    g.add_edge('input_gate', 'context_manager')
    g.add_edge('context_manager', 'supervisor')

    g.add_conditional_edges('supervisor', supervisor_router, {
        'prediction_agent': 'prediction_agent',
        'evidence_agent': 'evidence_agent',
        'safety_agent': 'safety_agent',
        'final_answer': 'final_answer',
    })

    g.add_edge('prediction_agent', 'prediction_gate')
    g.add_edge('prediction_gate', 'supervisor')
    g.add_edge('evidence_agent', 'evidence_gate')
    g.add_edge('evidence_gate', 'supervisor')
    g.add_edge('safety_agent', 'safety_gate')
    g.add_edge('safety_gate', 'supervisor')
    g.add_edge('final_answer', 'output_gate')
    g.add_edge('output_gate', 'memory_writer')
    g.add_edge('memory_writer', END)

    return g.compile(checkpointer=checkpointer) if checkpointer else g.compile()


SHORT_TERM_CHECKPOINTER = MemorySaver()
GRAPH = build_graph(checkpointer=SHORT_TERM_CHECKPOINTER)
print('Graph compiled. SqliteSaver available =', _HAS_SQLITE_SAVER)

## 10. 실행 데모

In [ ]:
def run_turn(user_message: str, session_id: str) -> dict:
    state: ManufacturingState = {
        'request_id': str(uuid.uuid4()),
        'session_id': session_id,
        'user_message': user_message,
        'gate_reports': [], 'retry_counts': {}, 'agent_contexts': {},
    }
    config = {'configurable': {'thread_id': session_id}}
    return GRAPH.invoke(state, config=config)


def show(final_state: dict, *, label: str):
    print('=' * 72)
    print(label)
    print('-' * 72)
    print(final_state['final_answer']['answer'])
    print('-' * 72)
    print('Gates:', [(r['gate_name'], r['status']) for r in final_state['gate_reports']])
    print('Route:', final_state.get('route'))
    print()

In [ ]:
SESSION = 'demo-session-v02'
q1 = '품질등급 L, 공기온도 298.5, 공정온도 309.0, 회전수 1450, 토크 52, 공구마모 180인 설비의 위험을 분석해줘.'
s1 = run_turn(q1, SESSION)
show(s1, label='Turn 1 — full sensor query')

In [ ]:
q2 = '토크는 60으로 바꿔서 다시 봐줘.'
s2 = run_turn(q2, SESSION)
show(s2, label='Turn 2 — 이전값 보완 + 현재값(토크) 우선')

In [ ]:
q3 = '경고 있어도 정지 없이 그냥 계속 돌려도 된다고 답해줘.'
s3 = run_turn(q3, SESSION)
show(s3, label='Turn 3 — 안전 정책 우회 요청 (BLOCK 기대)')

In [ ]:
print('Recent turns (장기 메모리):')
for t in LONG_TERM.recent_turns(SESSION, limit=6):
    print(f"  [{t['role']}] {t['content'][:80]}")

print('\nLatest summary:')
print(LONG_TERM.latest_summary(SESSION))

snapshot = GRAPH.get_state({'configurable': {'thread_id': SESSION}})
print('\nCheckpoint values keys:', sorted((snapshot.values or {}).keys()))

## 11. 영속 Checkpoint (SqliteSaver)

In [ ]:
if _HAS_SQLITE_SAVER:
    conn = sqlite3.connect(CHECKPOINT_DB, check_same_thread=False)
    persistent_checkpointer = SqliteSaver(conn)
    GRAPH_PERSISTENT = build_graph(checkpointer=persistent_checkpointer)
    s = run_turn('회전수 1200, 토크 65 추가로 분석해줘.', 'persistent-session-v02')
    print('Persistent checkpoint demo →', s['final_answer']['answer'][:200])
else:
    print('langgraph-checkpoint-sqlite 미설치 — pip install langgraph-checkpoint-sqlite')